<a href="https://colab.research.google.com/github/Farahani1/kodum/blob/claude%2Fsweet-franklin-pm0tdm/notebooks/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# kodoom on Colab

This notebook only connects Colab to the project: it mounts Drive, gets the code, installs it and runs `kodoom` commands. All logic lives in the repository, so what runs here is exactly what already ran on the laptop (`dev` profile).

**Every session:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

**After a session dies:** reconnect and Run all again. Logs and checkpoints are already on Drive; a run started again with the same run id resumes from its last checkpoint.

**To update the code:** push to the branch from the laptop, then re-run the *Get the code* cell (or Run all). See `docs/colab.md`.

## Settings

In [ ]:
REF = "claude/sweet-franklin-pm0tdm"  # branch, tag or commit to run
PROFILE = "colab-preflight"  # colab-preflight first, then colab
TOKEN_SECRET = "github-kodoom"  # name of the Colab secret holding the GitHub token
HF_TOKEN_SECRET = "HF_TOKEN"  # Colab secret with a Hugging Face read token (gated models)

## Mount Google Drive
Runs, logs and checkpoints are written to `MyDrive/kodoom/runs`. Nothing else goes to Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Get the code
Clones the repository the first time and moves to the latest `REF` every time it runs, discarding any edits made on Colab: code changes are made on the laptop and pushed.

For a private repository, add a GitHub token under 🔑 Secrets (left sidebar) named as `TOKEN_SECRET` in Settings (`github-kodoom`), with notebook access on. The token is used for each fetch and is not saved in the clone.

In [ ]:
import os
import subprocess

REPO = "github.com/Farahani1/kodum.git"
DIR = "/content/kodoom"

token, token_problem = None, None
try:
    from google.colab import userdata

    token = userdata.get(TOKEN_SECRET)
except Exception as e:  # a public repository needs no token
    token_problem = f"{type(e).__name__}: {e}"
url = f"https://x-access-token:{token}@{REPO}" if token else f"https://{REPO}"


def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        if not token and "Username" in message:
            message += (
                "\nThe repository is private and no token was used "
                f"({token_problem or TOKEN_SECRET + ' is empty'}). In Secrets (key icon, "
                f"left sidebar): name it exactly {TOKEN_SECRET}, paste the github_pat_... "
                "value, turn Notebook access on, then re-run this cell."
            )
        raise RuntimeError(message)  # git's own message, never the token
    return result.stdout.strip()


if not os.path.isdir(f"{DIR}/.git"):
    git("init", "--quiet", DIR)
git("-C", DIR, "fetch", "--quiet", "--depth", "50", url, REF)
git("-C", DIR, "checkout", "--quiet", "--force", "FETCH_HEAD")
print(git("-C", DIR, "log", "-1", "--format=%h %s (%cr)"))

In [ ]:
!pip install --quiet -e "{DIR}[colab]"

## Check the environment
Stops here if Drive is not mounted, no GPU is attached or Drive is too full. Also lists the runs that can be resumed.

In [ ]:
!kodoom check --profile {PROFILE}

In [ ]:
!kodoom runs --profile {PROFILE}

## Run
Pipeline steps are added here as they are built, each as one `kodoom` command with `--profile {PROFILE}`. Re-running a step with the same run id resumes it.

## Data

The skills data is generated, not downloaded: same seed, same files, in seconds. It is written to **your Drive** (`MyDrive/kodoom/data/skills`, see `data_dir` in the profile), never into the repository checkout, and nothing here pushes data anywhere. A manifest next to each file holds the seed and a checksum.

In [ ]:
!kodoom generate --profile {PROFILE}

## Hugging Face access (optional, needed for TranslateGemma)

Some models are gated. Once: open the model page, accept the license, create a **read** token at huggingface.co/settings/tokens, and add it under 🔑 Secrets as `HF_TOKEN` with notebook access on. This cell passes it to the commands below; without a secret everything else still works (with slower downloads).

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get(HF_TOKEN_SECRET)
    print("Hugging Face token set")
except Exception as e:  # not needed unless a model is gated
    print(f"no Hugging Face token ({type(e).__name__}); gated models will fail")

## Fetch typed-decisions (English originals)

Downloads `LocalLLaMA/typed-decisions` at its pinned commit and converts it to records on **your Drive** (`MyDrive/kodoom/data/.../typed-decisions/en`). Nothing is pushed anywhere. Preflight takes the first 200 cases per workflow and split; the `colab` profile takes all. **Copy the output of both cells back to the assistant.**

In [ ]:
!kodoom fetch typed-decisions --profile {PROFILE}

In [ ]:
!kodoom fields --profile {PROFILE}

## Fetch helmo/synthetic-typed-decisions (English)

The extra topic-breadth data (plan 1.1), converted to records on **your Drive** (`.../helmo/en`). Preflight takes the first 200 records; the `colab` profile takes all 9,879. **Copy the output back to the assistant.**

In [ ]:
!kodoom fetch helmo --profile {PROFILE}

## Translation trial (10 cases)

Translates 10 test cases with each candidate and prints the automatic checks and English next to Persian. Output goes to **your Drive** (`.../typed-decisions/fa/<translator>`); nothing is pushed. Needs the *Fetch typed-decisions* cell to have run. The first run of each model downloads it (several GB, into `/content/hf-cache`, never Drive). If a cell fails, **copy the error back**: the model ids and message format are not verified yet. **Copy the output of each cell back to the assistant.**

In [ ]:
T = "translategemma-4b"
!kodoom translate typed-decisions --profile {PROFILE} --translator {T} --split test --limit 10
!kodoom translations typed-decisions --profile {PROFILE} --translator {T} --show 3

In [ ]:
T = "qwen3-8b-4bit"
!kodoom translate typed-decisions --profile {PROFILE} --translator {T} --split test --limit 10
!kodoom translations typed-decisions --profile {PROFILE} --translator {T} --show 3

In [ ]:
!kodoom info --profile {PROFILE}

## Before closing (optional)
Everything is already on Drive as it is written. This forces Drive to finish syncing, for peace of mind before closing the tab.

In [ ]:
drive.flush_and_unmount()